# Generate KPIs

Team runner notebook. See **`README.md`** for full setup and config reference.

## Before you run

1. Upload **`config.py`** and the **`kpi_pipeline/`** folder next to this notebook.
2. Edit **`CONFIG`** in `config.py` (dates, scope mode, scope adjustments, input filters, slices, output save mode, HTML report).
3. Set **`run.mode`** — `full` (default) runs the pipeline; `html_only` loads saved Delta outputs and renders the HTML report only.
4. Run all cells top to bottom — Cell 2 previews inputs and the **Scope debug** cell reports distinct product/store counts per slice before the pipeline run (both skip when `run.mode=html_only`).

## What you get

- **`kpi_long`** — one tidy table for all metrics, periods, and slice dimensions
- **YoY / QoQ / WoW** comparisons (overall + each active slice)
- **Scope diff** — defined-only vs score-only scope (optional; `scope.run_scope_diff=True`)
- **Incremental Delta saves** — append missing periods; optional overwrite with warning
- **HTML report** — executive-style tabbed report: period → slice dimension → value tabs (`html_report.enabled=True` by default)

Pipeline logic lives in `kpi_pipeline/`; this notebook previews inputs, runs the pipeline (or loads saved outputs), previews the save plan, then writes outputs.


In [ ]:
# Cell 1 — Load config and print resolved settings (paths, date window, slices).
import sys
sys.path.insert(0, ".")

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
from algo_helpers import fundamentals as fund

%run ./config
settings = materialize(fund.paste)

from kpi_pipeline import KPIRunner
runner = KPIRunner(spark, settings)
runner.print_config_summary()


## Input previews (defined scope, lost sales, daily data)

Each table is read separately with the same **`input_filters`** from `config.py` that the pipeline uses.

Add ad-hoc filters here if needed (e.g. `.filter("brand = 'NIKE'")`) — config filters are for filters you want on every run.

Scope adjustment CSVs/files are **not** shown here; clean those yourself before enabling them in `scope_adjustments` (expected keys: `product_id`, `store_id`, date).

In [ ]:
# Cell 2 — Preview pipeline inputs (config filters applied automatically).
from kpi_pipeline.inputs import (
    preview_input_table,
    read_daily_data_source,
    read_defined_scope_source,
    read_lost_sales_source,
)

PREVIEW_LIMIT = 20

defined_scope_raw = read_defined_scope_source(spark, settings)
lost_sales_raw = read_lost_sales_source(spark, settings)
daily_data_raw = read_daily_data_source(spark, settings)

# Optional ad-hoc notebook filters (examples — uncomment to use):
# defined_scope_raw = defined_scope_raw.filter("store_id = 123")
# lost_sales_raw = lost_sales_raw.filter("product_id = 'SKU001'")
# daily_data_raw = daily_data_raw.filter("store_id NOT IN (829, 639, 917)")

_defined_cfg = settings["DEFINED_SCOPE"]
_defined_date_col = _defined_cfg.get("date_col")
if _defined_date_col is None:
    _defined_year_col = _defined_cfg.get("year_col")
    _defined_week_col = _defined_cfg.get("week_col")
    if _defined_year_col and _defined_week_col:
        print(
            f"defined_scope preview: native year/week path "
            f"(Year<-{_defined_year_col}, Week<-{_defined_week_col}) — no date_col filter"
        )
    else:
        print("defined_scope preview: no date_col or year/week columns configured — showing all rows")

print("=== defined_scope ===")
display(preview_input_table(
    defined_scope_raw, settings, "defined_scope", limit=PREVIEW_LIMIT,
    date_col=_defined_date_col,
))

print("=== lost_sales ===")
display(preview_input_table(
    lost_sales_raw, settings, "lost_sales", limit=PREVIEW_LIMIT, date_col="week_start_date",
))

if settings["LOST_SALES_ENSEMBLE_ENABLED"]:
    from kpi_pipeline.inputs import read_speed_cluster_source
    print("lost-sales ENSEMBLE enabled: fast (120d) previewed above; slow (365d) + speed-cluster below.")
    lost_sales_slow_raw = read_lost_sales_source(spark, settings, path=settings["PATH_LOST_SALES_SLOW"])
    speed_cluster_raw = read_speed_cluster_source(spark, settings)
    print("=== lost_sales (slow / 365-day) ===")
    display(preview_input_table(
        lost_sales_slow_raw, settings, "lost_sales_slow", limit=PREVIEW_LIMIT, date_col="week_start_date",
    ))
    print("=== speed_cluster (one row per product) ===")
    display(speed_cluster_raw.limit(PREVIEW_LIMIT))

print("=== daily_data ===")
display(preview_input_table(
    daily_data_raw, settings, "daily_data", limit=PREVIEW_LIMIT,
    date_col=settings["DAILY_TIME_COLUMNS"]["date"],
))


## Scope debug — product / store counts before the full run

Sanity-check scope **before** the heavy KPI computation in Cell 3. This recomputes scope independently (`runner.build_dimensions()` + `runner.build_scopes()`) and reports distinct `product_id`, `store_id`, and pair counts — **overall** and broken out by **each active slice dimension** (`slices` + `derived_dimensions` from products, and any enabled `dimension_sources`). The same `value_filters` the KPI step applies are applied here, so these counts match what `kpi_long` will report per slice.

This is a **read-only pre-flight check**. Cell 3's `runner.run()` rebuilds the same scope again as part of the full pipeline — the light duplication is intentional (mirrors how Cell 2 previews raw inputs before Cell 3 reads them), not a performance shortcut.

NULL slice values appear here as the literal string `"NULL"`; in `kpi_long` those same rows carry an empty/None `dimension_value`.

In [ ]:
# Scope debug — distinct product/store counts per slice (pre-flight, read-only).
# Recomputes scope independently; Cell 3 (runner.run) rebuilds the same scope.
if settings.get("RUN_MODE") == "html_only":
    print("html_only mode — scope debug skipped (no pipeline scope is built).")
else:
    runner.build_dimensions()
    runner.build_scopes(fund_paste=fund.paste)
    display(runner.scope_debug_summary())

In [ ]:
# Cell 3 — Run pipeline (compute only; save happens in later cells).
# When run.mode=html_only, loads saved Delta outputs instead of computing KPIs.
ctx = runner.run(fund_paste=fund.paste, save=False)
if settings.get("RUN_MODE") == "html_only":
    print("html_only mode — skipped input previews, scope summary, and save cells are not applicable.")


## Scope summary

Row counts by `scope_origin` in the **final scope** used for KPIs.

When scope adjustments are enabled, this section shows the scope **before** adjustments, each step, and the **after** final scope. Step-by-step logs also print during Cell 3 (runner.run).


In [ ]:
if ctx.scope_adjustments_applied:
    print("=== Scope BEFORE adjustments ===")
    display(runner.scope_before_adjustments_summary())
    print("=== Adjustment steps ===")
    display(runner.scope_adjustment_steps_table())
    print("=== Final scope AFTER adjustments ===")
else:
    print("No scope adjustments applied — showing final scope only.")

display(runner.hybrid_scope_summary())


## Save plan (when `save_outputs=True`)

Preview what will be appended vs skipped vs overwritten.

- **First-time backfill** (e.g. 2024 + 2025 + 2026): `save_mode='initial'`
- **Weekly refresh**: `save_mode='incremental'` with a narrow `run_min_date` for the new week only

If overlapping periods are skipped and you want to replace them, set `output.allow_overwrite_existing=True` in `config.py`, re-run cell 1, then re-run the save cell below.

In [ ]:
# Cell 4 — Preview incremental save plan (no writes yet).
from kpi_pipeline.io import save_outputs

save_plan = runner.preview_save_plan(fund.paste)


In [ ]:
# Cell 5 — Persist outputs after reviewing the save plan above.
if settings["SAVE_OUTPUTS"]:
    save_plan = save_outputs(ctx, fund.paste)
else:
    print("SAVE_OUTPUTS is False — nothing written.")

## kpi_long (sample)

Filter `ctx.kpi_long` by `period_type`, `period`, `dimension`, and `dimension_value` to build any panel.


In [ ]:
display(ctx.kpi_long.head(30))
if ctx.active_slice_dimensions:
    _ex = ctx.active_slice_dimensions[0]
    display(ctx.kpi_long[(ctx.kpi_long["period_type"] == "annual") & (ctx.kpi_long["dimension"] == _ex)])


## Comparisons (YoY / QoQ / WoW)


In [ ]:
from kpi_pipeline import slice_comparison_view

print("=== YoY — overall ===")
if ctx.yoy_display is not None and not ctx.yoy_display.empty:
    display(ctx.yoy_display)
else:
    print("skipped: needs >= 2 years of history.")

print("=== QoQ — overall ===")
if ctx.qoq_display is not None and not ctx.qoq_display.empty:
    display(ctx.qoq_display)
else:
    print("skipped: needs >= 2 quarters in sequence.")

print("=== MoM — overall (latest month vs prior month) ===")
if ctx.mom_display is not None and not ctx.mom_display.empty:
    display(ctx.mom_display)
else:
    print("skipped: needs >= 2 fiscal months.")

print("=== WoW — overall (latest week vs prior week) ===")
if ctx.wow_display is not None and not ctx.wow_display.empty:
    display(ctx.wow_display)
else:
    print("skipped: needs >= 2 fiscal weeks.")

for dim in ctx.active_slice_dimensions:
    print(f"=== YoY by slice: {dim} ===")
    yoy_slice = slice_comparison_view(ctx.comparison_yoy, dim)
    if not yoy_slice.empty:
        display(yoy_slice)
    else:
        print(f"skipped: no YoY rows for dimension '{dim}'.")

    print(f"=== QoQ by slice: {dim} ===")
    qoq_slice = slice_comparison_view(ctx.comparison_qoq, dim)
    if not qoq_slice.empty:
        display(qoq_slice)
    else:
        print(f"skipped: no QoQ rows for dimension '{dim}'.")

    print(f"=== MoM by slice: {dim} ===")
    mom_slice = slice_comparison_view(ctx.comparison_mom, dim)
    if not mom_slice.empty:
        display(mom_slice)
    else:
        print(f"skipped: no MoM rows for dimension '{dim}'.")

    print(f"=== WoW by slice: {dim} ===")
    wow_slice = slice_comparison_view(ctx.comparison_wow, dim)
    if not wow_slice.empty:
        display(wow_slice)
    else:
        print(f"skipped: no WoW rows for dimension '{dim}'.")


## Comparable pairs (like-for-like)

Only when `comparable_pairs.enabled=True`. For each comparison the metrics are recomputed over **only the `(product_id, store_id)` pairs present in both compared periods**. A comparison appears only when the run window spans both compared periods (e.g. a multi-year window for comparable YoY). Saved as `comparable_*` Delta tables and shown as a second comparison table per panel in the HTML report.

In [ ]:
# Comparable (like-for-like) comparisons — populated only when comparable_pairs.enabled=True.
if settings.get("COMPARABLE_PAIRS_ENABLED"):
    for lbl, disp in [
        ("YoY", ctx.comparable_yoy_display),
        ("QoQ", ctx.comparable_qoq_display),
        ("MoM", ctx.comparable_mom_display),
        ("WoW", ctx.comparable_wow_display),
    ]:
        print(f"=== Comparable {lbl} — overall (pairs present in BOTH periods) ===")
        if disp is not None and not disp.empty:
            display(disp)
        else:
            print("skipped: run window must span both compared periods for comparable pairs.")
    if ctx.comparable_kpi_long is not None and not ctx.comparable_kpi_long.empty:
        print("=== comparable_kpi_long (sample) ===")
        display(ctx.comparable_kpi_long.head(30))
else:
    print("Comparable pairs disabled — set comparable_pairs.enabled=True in config.py to enable.")


## Defined vs score scope diff

Annual key metrics under defined-only scope vs score-only scope. Large gaps may indicate missing defined coverage.

Requires `scope.run_scope_diff=True` in `config.py` (default `False`).


In [ ]:
if ctx.scope_diff is not None:
    display(ctx.scope_diff)
else:
    print("scope diff skipped — set scope.run_scope_diff=True in config.py to enable.")


## HTML report

Generates a standalone, offline HTML file with:
- **Period tabs** — Annual / Quarter / Weekly (horizontal)
- **Slice dimension tabs** — Overall plus every slice column found in `kpi_long` (inferred automatically)
- **Value tabs** — vertical sidebar within each slice dimension (one brand/category/etc. at a time)
- **KPI tables** — metrics as rows (colour-coded by category), periods as columns
- **Comparison section** — YoY / QoQ / WoW change table per value panel
- **Metric Details tab** — definition, store scope, and formula for every active metric
- **Executive header** — client, reporting window, scope mode, slice dimensions

Controlled by `html_report` in `config.py`. Set `enabled: False` to skip.

**HTML only from saved data:** set `run.mode: "html_only"` in `config.py` — loads `kpi_long` and comparison tables from saved Delta outputs without re-running the pipeline.

The file is written to the notebook's working directory by default.

In [ ]:
# Cell 6 — Generate HTML report (when html_report.enabled=True in config.py).
html_path = runner.build_html_report(local_dir=".")
if html_path:
    try:
        # On Databricks, displayHTML renders a clickable link.
        from IPython.display import HTML, display as ipy_display
        ipy_display(HTML(f'<p>HTML report: <a href="{html_path}" target="_blank">{html_path}</a></p>'))
    except Exception:
        print(f"HTML report: {html_path}")